# 08 · Recourse with an OOD gate (RQ3, empirical study)

**Question.** When counterfactual recourse for a GBDT must also pass a conformal OOD test, how much more plausible are the counterfactuals, and what does it cost in proximity, sparsity and success rate?

**Design (read before the numbers).**
- Model: LightGBM. Individuals: random test rows with P(y=1) < 0.5. Target: P(y=1) >= `target_prob`.
- Actionability (`configs/datasets/adult.yaml`, decision D16): age, sex, race, native-country, marital-status, relationship are immutable; spike features (capital-gain/loss) are not actionable; education-num can only increase.
- Methods (same search, same random seed per individual): `no_gate`, and `gate` = Isolation-Forest conformal rule (calibration-conditional, alpha/delta from `configs/recourse.yaml`).
- **The gate and the realism metrics use different detectors.** The gate is Isolation Forest; plausibility is measured with LOF (independent), calibrated on the same `calib` rows. A PASS of the gate does not mean in-distribution (the counterfactual is selected against the gate, see `recourse/gate.py`).
- One seed here. Repeat over seeds before quoting any number.

In [2]:
import numpy as np
import pandas as pd

from conformal_trees.data import load_dataset, make_splits
from conformal_trees.evaluation import flag_rate, paired_bootstrap_ci, robust_validity
from conformal_trees.models import GBDTClassifier
from conformal_trees.recourse import OODGate, RecourseSpace, SearchConfig, find_recourse, make_donors
from conformal_trees.signals import IsolationForestScore, LOFScore
from conformal_trees.utils import derive_seed, load_config, load_dataset_config, make_rng

pd.set_option("display.max_columns", 50)
cfg = load_config("base.yaml", "models.yaml", "recourse.yaml")
SEED = cfg["project"]["default_seed"]
N_INSTANCES = 500
N_ALT_MODELS = 3
ALPHA, DELTA = cfg["gate"]["alpha"], cfg["gate"]["delta"]
TARGET = cfg["recourse"]["target_prob"]

ds = load_dataset("adult")
sp = make_splits(ds.X, ds.y, cfg["splits"], seed=SEED)

## 1. Models, detectors, action space

In [3]:
model = GBDTClassifier.from_config(cfg["gbdt"], seed=SEED).fit(sp.train.X, sp.train.y)
alt_models = [GBDTClassifier.from_config(cfg["gbdt"], seed=derive_seed(SEED, "alt", i)).fit(sp.train.X, sp.train.y)
              for i in range(N_ALT_MODELS)]
iforest = IsolationForestScore(ds.numeric, ds.categorical, seed=SEED).fit(sp.train.X)
lof = LOFScore(ds.numeric, ds.categorical).fit(sp.train.X)

rc = load_dataset_config("adult")["recourse"]
space = RecourseSpace.from_dataset(
    ds, sp.train.X,
    extra_immutable=rc["extra_immutable"], monotone_up=rc["monotone_increasing"],
    spike_immutable=cfg["recourse"]["spike_immutable"], max_mode_share=cfg["recourse"]["max_mode_share"],
)
print("mutable features :", space.mutable)
print("not actionable (spike):", space.spiky)
print("monotone increasing  :", space.monotone_up)

gate = OODGate(iforest.score, iforest.score(sp.calib.X), alpha=ALPHA, delta=DELTA)
print(f"gate: k = {gate.k} of n_cal = {gate.n_cal}, score threshold = {gate.score_threshold:.4f}")
donors = make_donors(space, sp.train.X, sp.train.y)
search_cfg = SearchConfig.from_config(cfg)

d:\Development\projects for master\Paper researcher\Conformalized Tree-Based Gradient Ensembles for Out-of-Distribution Detection and Recourse\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\Development\projects for master\Paper researcher\Conformalized Tree-Based Gradient Ensembles for Out-of-Distribution Detection and Recourse\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\Development\projects for master\Paper researcher\Conformalized Tree-Based Gradient Ensembles for Out-of-Distribution Detection and Recourse\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprec

mutable features : ['education-num', 'hours-per-week', 'workclass', 'occupation']
not actionable (spike): ['capital-gain', 'capital-loss']
monotone increasing  : ['education-num']
gate: k = 439 of n_cal = 9769, score threshold = 0.5449


## 2. Pick individuals and run both methods (the slow cell)
Both methods get the same random seed for the same individual, so the comparison is paired.

In [4]:
p_test = model.predict_proba(sp.test.X)[:, 1]
neg = np.flatnonzero(p_test < 0.5)
rng = make_rng(derive_seed(SEED, "recourse", "pick"))
picked = rng.choice(neg, size=min(N_INSTANCES, len(neg)), replace=False)
print(f"{len(neg)} unfavourable test rows, using {len(picked)}")

runs = {"no_gate": [], "gate": []}
for i, idx in enumerate(picked):
    x = sp.test.X.iloc[idx]
    for name, g in (("no_gate", None), ("gate", gate)):
        runs[name].append(find_recourse(model, space, x, donors, cfg=search_cfg,
                                        rng=make_rng(derive_seed(SEED, "recourse", i)), gate=g))
print({k: sum(r.found for r in v) for k, v in runs.items()}, "found out of", len(picked))

7762 unfavourable test rows, using 500
{'no_gate': 307, 'gate': 294} found out of 500


## 3. Results per method
- `success`: share of individuals with a valid, actionable counterfactual that satisfies the method's own constraints.
- `cost`, `n_changed`: proximity (std units + categorical changes) and sparsity of the found counterfactuals.
- `own_flag`: share flagged by the **gate's own** detector (Isolation Forest) at alpha. For `no_gate` this shows what the gate would have rejected.
- `lof_flag`: share flagged by the **independent** detector (LOF) at alpha. In-distribution rows give about alpha (see the reference rows).
- `robust`: average share of 3 re-trained models (other seeds) for which the counterfactual still reaches the target.

In [5]:
def frame_of(results):
    found = [r for r in results if r.found]
    return space.to_frame({c: np.array([r.x_cf[c] for r in found], dtype=object if c in space.categorical else float)
                           for c in space.columns}), found

cal_lof = lof.score(sp.calib.X)
cal_if = iforest.score(sp.calib.X)
rows = {}
for name, results in runs.items():
    F, found = frame_of(results)
    rows[name] = {
        "success": len(found) / len(results),
        "cost": np.mean([r.cost for r in found]),
        "n_changed": np.mean([r.n_changed for r in found]),
        "own_flag": flag_rate(cal_if, iforest.score(F), ALPHA),
        "lof_flag": flag_rate(cal_lof, lof.score(F), ALPHA),
        "robust": robust_validity(alt_models, F, TARGET).mean(),
    }
table = pd.DataFrame(rows).T
table.round(3)

,success,cost,n_changed,own_flag,lof_flag,robust
no_gate,0.614,2.507,2.179,0.078,0.088,0.532
gate,0.588,2.460,2.163,0.024,0.092,0.533


In [6]:
# Reference rows for reading lof_flag: real favourable test rows, and the original unfavourable rows
real_pos = sp.test.X[(p_test >= 0.5)]
orig = sp.test.X.iloc[picked]
pd.Series({
    "real favourable test rows (n=%d)" % len(real_pos): flag_rate(cal_lof, lof.score(real_pos), ALPHA),
    "original unfavourable rows (n=%d)" % len(orig): flag_rate(cal_lof, lof.score(orig), ALPHA),
}, name="lof_flag").round(3).to_frame()

,lof_flag
real favourable test rows (n=2007),0.039
original unfavourable rows (n=500),0.048


## 4. Paired comparison on individuals where both methods succeed
`gate - no_gate`: mean difference with a 95% bootstrap CI. A positive `cost` / `n_changed` difference is the price of the gate.

In [7]:
from conformal_trees.conformal import conformal_pvalues

both = [i for i in range(len(picked)) if runs["no_gate"][i].found and runs["gate"][i].found]
print(f"{len(both)} individuals with both found")

def per_individual(results):
    sub = [results[i] for i in both]
    F, _ = frame_of(sub)
    return {
        "cost": np.array([r.cost for r in sub]),
        "n_changed": np.array([r.n_changed for r in sub], dtype=float),
        "own_flag": (conformal_pvalues(cal_if, iforest.score(F)) <= ALPHA).astype(float),
        "lof_flag": (conformal_pvalues(cal_lof, lof.score(F)) <= ALPHA).astype(float),
        "robust": robust_validity(alt_models, F, TARGET),
    }

g, n = per_individual(runs["gate"]), per_individual(runs["no_gate"])
rng_b = make_rng(derive_seed(SEED, "recourse", "boot"))

pd.DataFrame({
    m: dict(zip(["mean diff (gate - no_gate)", "ci low", "ci high"], paired_bootstrap_ci(g[m], n[m], rng=rng_b)))
    for m in g
}).T.round(3)

294 individuals with both found


,mean diff (gate - no_gate),ci low,ci high
cost,0.024,0.009,0.044
n_changed,0.020,0.003,0.041
own_flag,-0.014,-0.027,-0.003
lof_flag,0.003,0.000,0.010
robust,-0.002,-0.010,0.006


In [8]:
from conformal_trees.conformal import marginal_rank

n_cal = len(cal_if)
marg_thr = np.sort(cal_if)[n_cal - marginal_rank(n_cal, ALPHA)]       # threshold of the plain alpha-level test
id_band = float(((cal_if > marg_thr) & (cal_if <= gate.score_threshold)).mean())

F_gate, _ = frame_of(runs["gate"])
s_gate = iforest.score(F_gate)
cf_band = float(((s_gate > marg_thr) & (s_gate <= gate.score_threshold)).mean())

print(f"band between the plain alpha-test threshold and the gate threshold:")
print(f"  share of in-distribution calibration rows in it : {id_band:.4f}")
print(f"  share of gated counterfactuals in it            : {cf_band:.4f}  ({int(round(cf_band * len(s_gate)))} of {len(s_gate)})")

band between the plain alpha-test threshold and the gate threshold:
  share of in-distribution calibration rows in it : 0.0050
  share of gated counterfactuals in it            : 0.0238  (7 of 294)


## 6. Margin sweep: robustness versus cost
At `target_prob = 0.5` the counterfactuals sit on the decision boundary, so they survive re-training only about half of the time (`robust` ~ 0.53).
Here the search target is raised (0.5 -> 0.8) while `robust` is always measured against the 0.5 decision threshold of re-trained models.
Look at how `robust`, `success` and `cost` trade off, and whether the gate changes that trade-off.

In [9]:
from dataclasses import replace

TARGETS = (0.5, 0.6, 0.7, 0.8)
N_SWEEP = 150
rows = []
for t in TARGETS:
    cfg_t = replace(search_cfg, target_prob=t)
    for name, g in (("no_gate", None), ("gate", gate)):
        res = [find_recourse(model, space, sp.test.X.iloc[idx], donors, cfg=cfg_t,
                             rng=make_rng(derive_seed(SEED, "sweep", int(t * 100), i)), gate=g)
               for i, idx in enumerate(picked[:N_SWEEP])]
        F, found = frame_of(res)
        row = {"target_prob": t, "method": name, "success": len(found) / len(res)}
        if found:
            row.update({
                "cost": np.mean([r.cost for r in found]),
                "n_changed": np.mean([r.n_changed for r in found]),
                "own_flag": flag_rate(cal_if, iforest.score(F), ALPHA),
                "lof_flag": flag_rate(cal_lof, lof.score(F), ALPHA),
                "robust": robust_validity(alt_models, F, 0.5).mean(),   # always the 0.5 decision threshold
            })
        rows.append(row)
pd.DataFrame(rows).set_index(["target_prob", "method"]).round(3)

success   cost  n_changed  own_flag  lof_flag  robust
target_prob method                                                        
0.5         no_gate    0.593  2.353      2.135     0.124     0.079   0.629
            gate       0.553  2.248      2.084     0.048     0.084   0.614
0.6         no_gate    0.527  2.698      2.241     0.101     0.089   0.962
            gate       0.487  2.574      2.205     0.027     0.068   0.963
0.7         no_gate    0.447  2.853      2.507     0.104     0.090   1.000
            gate       0.420  2.775      2.476     0.032     0.095   1.000
0.8         no_gate    0.320  3.283      2.708     0.104     0.104   1.000
            gate       0.300  3.154      2.644     0.044     0.111   1.000

## 7. Does a gate help detectors it was not built for?
Three detectors: Isolation Forest (IF), LOF, kNN distance. Two gates: `gate_IF` and `gate_LOF` (same conformal rule, calibrated on `calib`). Every method is evaluated by **all three** detectors with the plain alpha-level conformal test (`flag_*`).
- A gate should lower the flag rate of **its own** detector (that is what it optimises).
- Evidence of real plausibility gain = the flag rate also falls for the **other** detectors. If it does not, the gate only learned its own detector.
- Reference rows (below the table) show what in-distribution rows and the original individuals look like to each detector.

In [10]:
from conformal_trees.signals import KNNScore

knn = KNNScore(ds.numeric, ds.categorical).fit(sp.train.X)
gate_lof = OODGate(lof.score, cal_lof, alpha=ALPHA, delta=DELTA)
evaluators = {"IF": (iforest.score, cal_if), "LOF": (lof.score, cal_lof), "kNN": (knn.score, knn.score(sp.calib.X))}

N_X = 200
rows = {}
for name, g in {"no_gate": None, "gate_IF": gate, "gate_LOF": gate_lof}.items():
    res = [find_recourse(model, space, sp.test.X.iloc[idx], donors, cfg=search_cfg,
                         rng=make_rng(derive_seed(SEED, "cross", i)), gate=g)
           for i, idx in enumerate(picked[:N_X])]
    F, found = frame_of(res)
    row = {"success": len(found) / len(res)}
    if found:
        row["cost"] = np.mean([r.cost for r in found])
        for ev, (fn, cal) in evaluators.items():
            row[f"flag_{ev}"] = flag_rate(cal, fn(F), ALPHA)
        row["robust"] = robust_validity(alt_models, F, 0.5).mean()
    rows[name] = row

ref = {}
for label, Xr in (("original unfavourable rows", sp.test.X.iloc[picked[:N_X]]), ("real favourable test rows", real_pos)):
    ref[label] = {f"flag_{ev}": flag_rate(cal, fn(Xr), ALPHA) for ev, (fn, cal) in evaluators.items()}
display(pd.DataFrame(rows).T.round(3))
pd.DataFrame(ref).T.round(3)

,success,cost,flag_IF,flag_LOF,flag_kNN,robust
no_gate,0.595,2.497,0.109,0.084,0.101,0.574
gate_IF,0.560,2.436,0.036,0.080,0.062,0.568
gate_LOF,0.590,2.492,0.110,0.042,0.076,0.590


,flag_IF,flag_LOF,flag_kNN
original unfavourable rows,0.050,0.025,0.055
real favourable test rows,0.077,0.039,0.071


## 5. Notes to carry into `docs/decisions.md`
- How large is `success` for each method? If it is low, the action space (D16) is too restrictive for the question; try relaxing `extra_immutable`.
- Does `lof_flag` of `gate` fall towards alpha compared with `no_gate`, and by how much compared with the reference rows? An improvement only on `own_flag` is not evidence of realism (the search optimised against that detector).
- Price of the gate: paired `cost` / `n_changed` differences and the change in `success` and `robust`.
- Required before any claim: repeat over the 10 seeds, add baselines that exist in the literature (DiCE, OCEAN, CONFEX: check availability and solver licences first), and a second dataset.